In [1]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

In [2]:
df = pd.read_csv("data/sales_data_2.csv")

In [3]:
df.shape

(76000, 16)

In [4]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(" ", "_")
df['date'] = pd.to_datetime(df['date'])

In [5]:
# defining the target and column types
TARGET = "demand"

categorical_cols = [
    "store_id", "product_id", "category", "region",
    "weather_condition", "seasonality"
]

for col in categorical_cols:
    df[col] = df[col].astype("category")

In [6]:
# Convert discount percent → decimal
df["discount"] = df["discount"] / 100.0

# Final selling price after discount
df["final_price"] = df["price"] * (1 - df["discount"])

# Intuitive discount flag
df["has_discount"] = (df["discount"] > 0).astype(int)


In [7]:
df.sort_values(["store_id", "product_id", "date"])

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,price,discount,weather_condition,promotion,competitor_pricing,seasonality,epidemic,demand,final_price,has_discount
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,0.05,Snowy,0,85.73,Winter,0,115,69.0840,1
100,2022-01-02,S001,P0001,Electronics,North,93,71,0,65.63,0.05,Snowy,0,73.66,Winter,0,84,62.3485,1
200,2022-01-03,S001,P0001,Electronics,North,274,142,229,68.55,0.15,Snowy,1,80.73,Winter,0,132,58.2675,1
300,2022-01-04,S001,P0001,Electronics,North,132,42,0,61.66,0.10,Snowy,0,54.88,Winter,0,67,55.4940,1
400,2022-01-05,S001,P0001,Electronics,North,319,129,0,59.56,0.25,Snowy,1,57.34,Winter,0,110,44.6700,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75599,2024-01-26,S005,P0020,Toys,North,99,99,133,34.99,0.00,Cloudy,0,37.50,Winter,0,134,34.9900,0
75699,2024-01-27,S005,P0020,Toys,North,133,28,0,22.55,0.10,Snowy,0,26.95,Winter,0,38,20.2950,1
75799,2024-01-28,S005,P0020,Toys,North,105,83,122,30.87,0.15,Cloudy,1,28.08,Winter,0,130,26.2395,1
75899,2024-01-29,S005,P0020,Toys,North,144,112,94,31.95,0.05,Cloudy,0,31.33,Winter,0,105,30.3525,1


In [8]:
# creating time features
df["day"] = df["date"].dt.day
df["weekday"] = df["date"].dt.weekday
df["week"] = df["date"].dt.isocalendar().week.astype(int)
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year
df["is_weekend"] = df["weekday"].isin([5, 6]).astype(int)

In [9]:
# creating lag features
group_cols = ["store_id", "product_id"]
def create_lags(df, group_cols, target="demand"):
    df["lag_1"] = df.groupby(group_cols)[target].shift(1)
    df["lag_7"] = df.groupby(group_cols)[target].shift(7)
    df["lag_14"] = df.groupby(group_cols)[target].shift(14)
    df["lag_30"] = df.groupby(group_cols)[target].shift(30)
    df["lag_60"] = df.groupby(group_cols)[target].shift(60)
    return df

df = create_lags(df, group_cols)

C:\Users\bhish\AppData\Local\Temp\ipykernel_22680\316864207.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["lag_1"] = df.groupby(group_cols)[target].shift(1)
C:\Users\bhish\AppData\Local\Temp\ipykernel_22680\316864207.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["lag_7"] = df.groupby(group_cols)[target].shift(7)
C:\Users\bhish\AppData\Local\Temp\ipykernel_22680\316864207.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and sile

In [11]:
# rolling window feature

df["rolling_mean_7"] = (
    df.groupby(group_cols, observed=False)["demand"]
      .shift(1).rolling(7).mean()
)

df["rolling_mean_30"] = (
    df.groupby(group_cols,  observed=False)["demand"]
      .shift(1).rolling(30).mean()
)

df["rolling_std_30"] = (
    df.groupby(group_cols,  observed=False)["demand"]
      .shift(1).rolling(30).std()
)

# Drop rows with NaN lags
df = df.dropna()

In [12]:
# train-validation split
# split_date = "2023-12-01"
train = df[df["date"] < "2023-12-01"]
valid = df[(df["date"] >= "2023-12-01") & (df["date"] <= "2024-01-30")]

In [13]:
print("Train size:", len(train))
print("Valid size:", len(valid))

Train size: 63771
Valid size: 6100


In [14]:
df.columns

Index(['date', 'store_id', 'product_id', 'category', 'region',
       'inventory_level', 'units_sold', 'units_ordered', 'price', 'discount',
       'weather_condition', 'promotion', 'competitor_pricing', 'seasonality',
       'epidemic', 'demand', 'final_price', 'has_discount', 'day', 'weekday',
       'week', 'month', 'year', 'is_weekend', 'lag_1', 'lag_7', 'lag_14',
       'lag_30', 'lag_60', 'rolling_mean_7', 'rolling_mean_30',
       'rolling_std_30'],
      dtype='object')

In [15]:
features = [
    # ---- Categorical Metadata ----
    "store_id", "product_id", "category", "region",
    "weather_condition", "seasonality",

    # ---- Numeric Features ----
    "inventory_level", "units_sold", "units_ordered",
    "price", "final_price", "discount", "has_discount",
    "promotion", "competitor_pricing",
    "epidemic",

    # ---- Time Features ----
    "day", "weekday", "week", "month", "year", "is_weekend",

    # ---- Lag Features ----
    "lag_1", "lag_7", "lag_14", "lag_30", "lag_60",

    # ---- Rolling Features ----
    "rolling_mean_7", "rolling_mean_30", "rolling_std_30"
]

In [21]:
# initializing LGBMRegressor
model = LGBMRegressor(
    objective="regression",
    num_leaves=64,
    learning_rate=0.03,
    n_estimators=5000,
    feature_fraction=0.9,
    bagging_fraction=0.8,
    bagging_freq=4,
    min_data_in_leaf=50,
    verbose=100,
    early_stopping_rounds=200,
    n_jobs=-1   # use all CPU cores
)

In [22]:
# training the model
model.fit(
    train[features],
    train[TARGET],
    eval_set=[(valid[features], valid[TARGET])],
    eval_metric="mae",
    categorical_feature=categorical_cols
)

[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] early_stopping_round is set=200, early_stopping_rounds=200 will be ignored. Current value: early_stopping_round=200
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_

,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.03
,n_estimators,5000
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [23]:
valid_pred = model.predict(valid[features])
mae = mean_absolute_error(valid[TARGET], valid_pred)

print("Validation MAE:", mae)

[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
Validation MAE: 5.931192217967184
